# Drive mounting

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Extraction from Google drive which was donwloaded by the kaggle link & Verification

In [2]:
import os

# paths
drive_zip_path = '/content/drive/MyDrive/ANTLINGS_Drone_CV/dataset/archive.zip'
local_extract_path = '/content/visdrone_dataset'

# Create the local folder
os.makedirs(local_extract_path, exist_ok=True)

# Unzip quietly (-q) to save output space
print("Extracting dataset to local Colab storage... This might take a minute.")
!unzip -q "{drive_zip_path}" -d "{local_extract_path}"
print("Extraction complete!")

Extracting dataset to local Colab storage... This might take a minute.
Extraction complete!


In [4]:
!ls -lh "{local_extract_path}"

total 4.0K
drwxr-xr-x 6 root root 4.0K May 14 00:49 VisDrone_Dataset


# YAML Configuration & ultralytics installation

In [5]:
!find /content/visdrone_dataset -maxdepth 2 -type d

print("\n--- Searching for YAML Files ---")
!find /content/visdrone_dataset -name "*.yaml"

/content/visdrone_dataset
/content/visdrone_dataset/VisDrone_Dataset
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-val
/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-challenge

--- Searching for YAML Files ---
/content/visdrone_dataset/VisDrone_Dataset/visdrone.yaml


In [7]:
import yaml

old_yaml_path = '/content/visdrone_dataset/VisDrone_Dataset/visdrone.yaml'
new_yaml_path = '/content/ants_visdrone.yaml'

try:
    # the original YAML provided by the Kaggle author
    with open(old_yaml_path, 'r') as f:
        data = yaml.safe_load(f)


    data['train'] = '/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train/images'
    data['val'] = '/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-val/images'
    data['test'] = '/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/images'

    # new production YAML
    with open(new_yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)

    print(f"Production YAML successfully created at: {new_yaml_path}")
    print("\n--- YAML Contents ---")


    with open(new_yaml_path, 'r') as f:
        print(f.read())

except Exception as e:
    print(f"Error: {e}")

Production YAML successfully created at: /content/ants_visdrone.yaml

--- YAML Contents ---
path: ./VisDrone_Dataset
train: /content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train/images
val: /content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-val/images
test: /content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/images
nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor



In [8]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00


# Visual Sanity Check  with Grid and Names

In [10]:
import os
import glob
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics.utils.plotting import Annotator

# 1. Map our class IDs to the actual names from your YAML
class_names = {
    0: "pedestrian", 1: "people", 2: "bicycle", 3: "car",
    4: "van", 5: "truck", 6: "tricycle", 7: "awning-tricycle",
    8: "bus", 9: "motor"
}

# 2. Grab 4 random images to create a grid
train_images = glob.glob('/content/visdrone_dataset/VisDrone_Dataset/VisDrone2019-DET-train/images/*.jpg')
random_images = random.sample(train_images, 4)

plt.figure(figsize=(18, 14))

# 3. Loop through and plot them
for i, img_path in enumerate(random_images):
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    annotator = Annotator(img, line_width=2) # Thinner lines for tiny objects

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                cls_id, x, y, w, h = map(float, line.split())

                # Fetch the string name, fallback to ID if something is weird
                cls_name = class_names.get(int(cls_id), f"Unknown({int(cls_id)})")

                # YOLO formats are normalized (0 to 1). Convert back to pixels.
                ih, iw, _ = img.shape
                x1 = int((x - w/2) * iw)
                y1 = int((y - h/2) * ih)
                x2 = int((x + w/2) * iw)
                y2 = int((y + h/2) * ih)

                # Plot with the actual name
                annotator.box_label([x1, y1, x2, y2], label=cls_name)

    plt.subplot(2, 2, i+1)
    plt.imshow(annotator.result())
    plt.axis('off')
    plt.title(f"Sample {i+1}: {os.path.basename(img_path)}")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.